# Sección nueva

In [2]:
!pip install -q transformers torch pandas tqdm

In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm
from google.colab import files

# ==============================
# 1. CONFIGURACIÓN INICIAL
# ==============================

# Nombre de tu archivo limpio
input_file = "reviews_clean_emoji_text.csv"

# Cargar archivo
df = pd.read_csv(input_file)

print("Archivo cargado correctamente.")
print("Número total de filas:", len(df))
print("Número total de columnas:", len(df.columns))

# Mostrar primeras filas
display(df.head())

# ==============================
# 2. VALIDAR COLUMNAS NECESARIAS
# ==============================

required_columns = ["review_id", "place_id", "review_text_clean"]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Faltan estas columnas en el archivo: {missing_columns}")
else:
    print("Todo bien. Las columnas necesarias existen.")

# ==============================
# 3. REVISIÓN GENERAL DEL TEXTO
# ==============================

print("\nInformación general del dataframe:")
df.info()

print("\nReviews vacías en review_text_clean:", df["review_text_clean"].isna().sum())

print("\nEjemplos de comentarios limpios:")
display(df[["review_id", "place_id", "review_text_clean"]].head(10))

Archivo cargado correctamente.
Número total de filas: 53280
Número total de columnas: 18


,review_id,source_row,place_id,name,rating,user_ratings_total,review_rating,review_text,review_text_clean,has_review_text,text_length,word_count,is_short_review,clean_text_is_empty,has_url,has_numbers,has_emoji,emoji_count
0,1,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,La Casa Del Pastor Perisur,4.2,3321.0,4,Los tacos de pastor “rojo” están ricos pero lo...,los tacos de pastor “rojo” están ricos pero lo...,True,489,92,False,False,False,True,False,0
1,2,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,La Casa Del Pastor Perisur,4.2,3321.0,5,Esta sucursal en especial me parece fenomenal....,esta sucursal en especial me parece fenomenal....,True,226,36,False,False,False,False,True,1
2,3,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,La Casa Del Pastor Perisur,4.2,3321.0,5,"Riquísimos y deliciosos tacos, desde Pastor tr...","riquísimos y deliciosos tacos, desde pastor tr...",True,431,64,False,False,False,False,False,0
3,4,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,La Casa Del Pastor Perisur,4.2,3321.0,4,La comida es buena (a secas) tiene buen sabor ...,la comida es buena (a secas) tiene buen sabor ...,True,307,58,False,False,False,False,False,0
4,5,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,La Casa Del Pastor Perisur,4.2,3321.0,5,La casa del pastor se encuentra ubicado en pla...,la casa del pastor se encuentra ubicado en pla...,True,447,81,False,False,False,False,False,0


Todo bien. Las columnas necesarias existen.

Información general del dataframe:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53280 entries, 0 to 53279
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   review_id            53280 non-null  int64  
 1   source_row           53280 non-null  int64  
 2   place_id             53280 non-null  object 
 3   name                 53275 non-null  object 
 4   rating               53216 non-null  float64
 5   user_ratings_total   53216 non-null  float64
 6   review_rating        53280 non-null  int64  
 7   review_text          53280 non-null  object 
 8   review_text_clean    53280 non-null  object 
 9   has_review_text      53280 non-null  bool   
 10  text_length          53280 non-null  int64  
 11  word_count           53280 non-null  int64  
 12  is_short_review      53280 non-null  bool   
 13  clean_text_is_empty  53280 non-null  bool   
 14  has_ur

,review_id,place_id,review_text_clean
0,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,los tacos de pastor “rojo” están ricos pero lo...
1,2,ChIJ5eXcxHcAzoUR49qRBI1-qGc,esta sucursal en especial me parece fenomenal....
2,3,ChIJ5eXcxHcAzoUR49qRBI1-qGc,"riquísimos y deliciosos tacos, desde pastor tr..."
3,4,ChIJ5eXcxHcAzoUR49qRBI1-qGc,la comida es buena (a secas) tiene buen sabor ...
4,5,ChIJ5eXcxHcAzoUR49qRBI1-qGc,la casa del pastor se encuentra ubicado en pla...
5,6,ChIJhwPkeGgAzoURByMi2OzdEX0,la comida ya no es buena.pedi flautas de papa ...
6,7,ChIJhwPkeGgAzoURByMi2OzdEX0,"que tal mochileros, está ves tuvimos una mala ..."
7,8,ChIJhwPkeGgAzoURByMi2OzdEX0,"ambiente familiar, tienda con amplia variedad ..."
8,9,ChIJhwPkeGgAzoURByMi2OzdEX0,lamentable el acoso del cajero en la hora de l...
9,10,ChIJhwPkeGgAzoURByMi2OzdEX0,"no inventen, nunca había probado los pasteles ..."


In [4]:
# ==============================
# 4. CARGAR MODELO TRANSFORMER
# ==============================

# Detectar si Colab tiene GPU
device = 0 if torch.cuda.is_available() else -1

print("GPU disponible:", torch.cuda.is_available())

# Modelo de sentimiento en español
sentiment_model_name = "pysentimiento/robertuito-sentiment-analysis"

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=sentiment_model_name,
    tokenizer=sentiment_model_name,
    device=device
)

print("Modelo cargado correctamente:", sentiment_model_name)

# ==============================
# 5. PRUEBA CON POCOS COMENTARIOS
# ==============================

sample_texts = df["review_text_clean"].head(10).astype(str).tolist()

sample_results = sentiment_pipeline(
    sample_texts,
    truncation=True,
    max_length=128
)

for text, result in zip(sample_texts, sample_results):
    print("Review:", text)
    print("Resultado:", result)
    print("-" * 80)

GPU disponible: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Modelo cargado correctamente: pysentimiento/robertuito-sentiment-analysis
Review: los tacos de pastor “rojo” están ricos pero los de pastor negro tienen un sabor muy fuerte, recomiendo que pidan uno y uno para probar al inicio. hay muy poco espacio entre cada mesa y la gente te empuja (no por descortesía, por que no hay espacio). al momento de ir nos tocó esperar 5 minutos aprox en la entrada y la orden tardó unos 10 minutos. es buen tiempo de entrega y atención. hay meseros amables y atentos y otros que tienen mala actitud. pero en general la experiencia fue buena
Resultado: {'label': 'POS', 'score': 0.5847817063331604}
--------------------------------------------------------------------------------
Review: esta sucursal en especial me parece fenomenal. los platillos están bien servidos y la relación calidad precio es bastante buena. lo recomiendo mucho. por favor no se pierdan los tacos de pastor negro, una delicia. [emoji_taco]
Resultado: {'label': 'POS', 'score': 0.9817201495170593

In [5]:
# ==============================
# 6. APLICAR SENTIMIENTO A TODAS LAS REVIEWS
# ==============================

texts = df["review_text_clean"].astype(str).tolist()

# Puedes usar 64. Si Colab se traba, cambia a 32.
batch_size = 64

sentiment_labels = []
sentiment_scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]

    results = sentiment_pipeline(
        batch_texts,
        truncation=True,
        max_length=128
    )

    for result in results:
        sentiment_labels.append(result["label"])
        sentiment_scores.append(result["score"])

print("Listo. Resultados generados:", len(sentiment_labels))
print("Filas originales:", len(df))

# Validar que coincidan las longitudes
if len(sentiment_labels) != len(df):
    raise ValueError("El número de resultados no coincide con el número de filas.")
else:
    print("Todo bien. Cada review tiene un resultado de sentimiento.")


# ==============================
# 7. AGREGAR RESULTADOS AL DATAFRAME
# ==============================

df["sentiment_label"] = sentiment_labels
df["sentiment_score"] = sentiment_scores

# Convertir sentimiento a variable numérica
# POS = 1, NEU = 0, NEG = -1

sentiment_map = {
    "POS": 1,
    "NEU": 0,
    "NEG": -1
}

df["sentiment_numeric"] = df["sentiment_label"].map(sentiment_map)

# Revisar resultados
display(
    df[[
        "review_id",
        "place_id",
        "review_text_clean",
        "sentiment_label",
        "sentiment_score",
        "sentiment_numeric"
    ]].head(20)
)

  0%|          | 0/833 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Listo. Resultados generados: 53280
Filas originales: 53280
Todo bien. Cada review tiene un resultado de sentimiento.


,review_id,place_id,review_text_clean,sentiment_label,sentiment_score,sentiment_numeric
0,1,ChIJ5eXcxHcAzoUR49qRBI1-qGc,los tacos de pastor “rojo” están ricos pero lo...,POS,0.584782,1
1,2,ChIJ5eXcxHcAzoUR49qRBI1-qGc,esta sucursal en especial me parece fenomenal....,POS,0.981720,1
2,3,ChIJ5eXcxHcAzoUR49qRBI1-qGc,"riquísimos y deliciosos tacos, desde pastor tr...",POS,0.943110,1
3,4,ChIJ5eXcxHcAzoUR49qRBI1-qGc,la comida es buena (a secas) tiene buen sabor ...,POS,0.949377,1
4,5,ChIJ5eXcxHcAzoUR49qRBI1-qGc,la casa del pastor se encuentra ubicado en pla...,POS,0.972359,1
5,6,ChIJhwPkeGgAzoURByMi2OzdEX0,la comida ya no es buena.pedi flautas de papa ...,NEG,0.929104,-1
6,7,ChIJhwPkeGgAzoURByMi2OzdEX0,"que tal mochileros, está ves tuvimos una mala ...",NEG,0.632223,-1
7,8,ChIJhwPkeGgAzoURByMi2OzdEX0,"ambiente familiar, tienda con amplia variedad ...",POS,0.774046,1
8,9,ChIJhwPkeGgAzoURByMi2OzdEX0,lamentable el acoso del cajero en la hora de l...,NEG,0.959621,-1
9,10,ChIJhwPkeGgAzoURByMi2OzdEX0,"no inventen, nunca había probado los pasteles ...",POS,0.849529,1


In [6]:
# ==============================
# 8. DISTRIBUCIÓN DE SENTIMIENTOS
# ==============================

print("Conteo de sentimientos:")
display(df["sentiment_label"].value_counts())

print("\nPorcentaje de sentimientos:")
display(df["sentiment_label"].value_counts(normalize=True) * 100)

print("\nPromedio general de sentimiento numérico:")
print(df["sentiment_numeric"].mean())

# ==============================
# 9. EJEMPLOS POR CLASE
# ==============================

for label in ["POS", "NEU", "NEG"]:
    print(f"\nEjemplos de reviews clasificadas como {label}:")

    examples = df[df["sentiment_label"] == label][
        ["review_text_clean", "sentiment_label", "sentiment_score"]
    ].head(10)

    display(examples)

Conteo de sentimientos:


,count
sentiment_label,
POS,34893
NEG,13440
NEU,4947



Porcentaje de sentimientos:


,proportion
sentiment_label,
POS,65.489865
NEG,25.225225
NEU,9.284910



Promedio general de sentimiento numérico:
0.4026463963963964

Ejemplos de reviews clasificadas como POS:


,review_text_clean,sentiment_label,sentiment_score
0,los tacos de pastor “rojo” están ricos pero lo...,POS,0.584782
1,esta sucursal en especial me parece fenomenal....,POS,0.981720
2,"riquísimos y deliciosos tacos, desde pastor tr...",POS,0.943110
3,la comida es buena (a secas) tiene buen sabor ...,POS,0.949377
4,la casa del pastor se encuentra ubicado en pla...,POS,0.972359
7,"ambiente familiar, tienda con amplia variedad ...",POS,0.774046
9,"no inventen, nunca había probado los pasteles ...",POS,0.849529
11,"la comida exquisita a mi parecer, pedí ramen y...",POS,0.977105
15,muy rico siempre buena opción,POS,0.930683
16,"el sabor de la comida es muy rico, la comida l...",POS,0.977664



Ejemplos de reviews clasificadas como NEU:


,review_text_clean,sentiment_label,sentiment_score
42,"amplia variedad de alimentos, desayunos, comid...",NEU,0.436842
60,quiero expresar mi inconformidad por el trato ...,NEU,0.672631
100,marcan como abierto pero todavia no abre. cuan...,NEU,0.850616
107,la comida no está mal aunque nada memorable. p...,NEU,0.523129
111,#¿nombre?,NEU,0.814733
198,ricos,NEU,0.533543
206,si vienes a la cdmx tienes que comerlos ! los ...,NEU,0.472410
240,alguien sabe si todos los puestos de ahí abren...,NEU,0.838768
250,buenas noches pasé a la tienda hoy a las 9:00p...,NEU,0.573313
259,abril 2025: una sucursal más de esta cadena de...,NEU,0.696402



Ejemplos de reviews clasificadas como NEG:


,review_text_clean,sentiment_label,sentiment_score
5,la comida ya no es buena.pedi flautas de papa ...,NEG,0.929104
6,"que tal mochileros, está ves tuvimos una mala ...",NEG,0.632223
8,lamentable el acoso del cajero en la hora de l...,NEG,0.959621
10,"la misoshiro sabe horrible, como si le hubiera...",NEG,0.982645
12,"pésimo servicio. pedí para llevar y, además de...",NEG,0.976635
13,consumí sopa miso con limón esa sopa no lleva ...,NEG,0.978766
14,la comida es buena sin duda [emoji_cara_feliz_...,NEG,0.785327
19,la pasta ni sabor tiene [emoji_gato_llorando] ...,NEG,0.676234
21,la atención es buena ya que los meseros son mu...,NEG,0.853816
22,por primera vez me invitaron a este restaurant...,NEG,0.909886


In [7]:
# ==============================
# 10. GUARDAR ARCHIVO COMPLETO
# ==============================

output_file = "reviews_with_transformer_sentiment.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Archivo guardado:", output_file)

files.download(output_file)

# ==============================
# 11. GUARDAR ARCHIVO REDUCIDO
# ==============================

sentiment_columns = [
    "review_id",
    "place_id",
    "name",
    "review_rating",
    "review_text",
    "review_text_clean",
    "text_length",
    "word_count",
    "has_emoji",
    "emoji_count",
    "sentiment_label",
    "sentiment_score",
    "sentiment_numeric"
]

# Validar que existan las columnas antes de seleccionarlas
existing_sentiment_columns = [
    col for col in sentiment_columns if col in df.columns
]

df_sentiment = df[existing_sentiment_columns].copy()

output_file_light = "reviews_sentiment_only.csv"

df_sentiment.to_csv(output_file_light, index=False, encoding="utf-8-sig")

print("Archivo guardado:", output_file_light)
print("Columnas guardadas:")
print(existing_sentiment_columns)



Archivo guardado: reviews_with_transformer_sentiment.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo guardado: reviews_sentiment_only.csv
Columnas guardadas:
['review_id', 'place_id', 'name', 'review_rating', 'review_text', 'review_text_clean', 'text_length', 'word_count', 'has_emoji', 'emoji_count', 'sentiment_label', 'sentiment_score', 'sentiment_numeric']


In [8]:
# ==============================
# 12. RESUMEN AUTOMÁTICO DE SENTIMIENTO
# ==============================

total_reviews = len(df)
sentiment_counts = df["sentiment_label"].value_counts()
sentiment_percentages = df["sentiment_label"].value_counts(normalize=True) * 100
avg_sentiment = df["sentiment_numeric"].mean()

print("RESUMEN DE ANÁLISIS DE SENTIMIENTO")
print("=" * 60)
print("Total de reviews analizadas:", total_reviews)
print("\nConteo por sentimiento:")
print(sentiment_counts)
print("\nPorcentaje por sentimiento:")
print(sentiment_percentages.round(2))
print("\nPromedio general de sentimiento numérico:", round(avg_sentiment, 4))

RESUMEN DE ANÁLISIS DE SENTIMIENTO
Total de reviews analizadas: 53280

Conteo por sentimiento:
sentiment_label
POS    34893
NEG    13440
NEU     4947
Name: count, dtype: int64

Porcentaje por sentimiento:
sentiment_label
POS    65.49
NEG    25.23
NEU     9.28
Name: proportion, dtype: float64

Promedio general de sentimiento numérico: 0.4026


In [9]:
#REVISIÓN


#Para ver si se perdieron reviews

print("Filas en df:", len(df))
print("Columnas:", df.shape[1])

# Si esperas 53280:
if len(df) == 53280:
    print("Bien: se conservan las 53,280 reviews.")
else:
    print("Revisar: el número de filas no coincide con lo esperado.")

#Revisar si no hay nulos en las columnas nuevas
cols_sentiment = [
    "sentiment_label",
    "sentiment_score",
    "sentiment_numeric"
]

print(df[cols_sentiment].isna().sum())


#Revisar etiquetas

print(df["sentiment_label"].unique())
print(df["sentiment_label"].value_counts())


#Revisar porcentajes de sentimiento

sentiment_percentages = df["sentiment_label"].value_counts(normalize=True) * 100
print(sentiment_percentages.round(2))


#Revisar rango del score
print("Score mínimo:", df["sentiment_score"].min())
print("Score máximo:", df["sentiment_score"].max())

df["sentiment_score"].describe()


#Revisar ejemplos aleatorios

for label in ["POS", "NEU", "NEG"]:
    print(f"\nEjemplos aleatorios de {label}:")
    display(
        df[df["sentiment_label"] == label][
            ["review_text_clean", "sentiment_label", "sentiment_score", "sentiment_numeric"]
        ].sample(10, random_state=42)
    )


#Revisar casos de baja confianza
df_low_confidence = df.sort_values("sentiment_score").head(30)

display(
    df_low_confidence[
        ["review_text_clean", "sentiment_label", "sentiment_score", "sentiment_numeric"]
    ]
)

Filas en df: 53280
Columnas: 21
Bien: se conservan las 53,280 reviews.
sentiment_label      0
sentiment_score      0
sentiment_numeric    0
dtype: int64
['POS' 'NEG' 'NEU']
sentiment_label
POS    34893
NEG    13440
NEU     4947
Name: count, dtype: int64
sentiment_label
POS    65.49
NEG    25.23
NEU     9.28
Name: proportion, dtype: float64
Score mínimo: 0.34082716703414917
Score máximo: 0.9893592596054077

Ejemplos aleatorios de POS:


,review_text_clean,sentiment_label,sentiment_score,sentiment_numeric
37511,100% recomendable. me ha maquillado varías vec...,POS,0.975966,1
24636,el mejor skatepark de la cdmx. muy bien ubicad...,POS,0.928051,1
25381,"buena atención, buenas mercancías aunque muy p...",POS,0.771347,1
11633,excelente atención y repartidores muy rápidos,POS,0.960104,1
7691,"el lugar es super lindo y limpio, mar es muy a...",POS,0.944541,1
46219,mi hija asistió ahí de los 2 a los 5 años y fu...,POS,0.979916,1
37423,la marca es clásica los productos de primera,POS,0.757413,1
45019,"el trato muy atento y amable, la señorita que ...",POS,0.966358,1
46146,si estás buscando hacerte algún tratamiento pa...,POS,0.920089,1
7496,el doctor muy amable.,POS,0.967000,1



Ejemplos aleatorios de NEU:


,review_text_clean,sentiment_label,sentiment_score,sentiment_numeric
1817,he acudido a esta estética por su buena atenci...,NEU,0.534606,0
8679,una tienda bastante surtida,NEU,0.577088,0
6785,deliciosa agua de coco [emoji_pulgar_hacia_arr...,NEU,0.620497,0
42635,"hola soy rodolfo ibarra, yo deje mi motociclet...",NEU,0.875183,0
42872,"los tacos estan sabrosos, pero la dirección es...",NEU,0.582510,0
18675,el pan es rico igual a los pasteles individual...,NEU,0.594776,0
6101,cuándo será la inauguración?,NEU,0.890497,0
49232,"muy atentos y responsables, las entregas son e...",NEU,0.589394,0
16211,se ubica dentro del centro medico coyoacan 2do...,NEU,0.839337,0
44144,"un oxxo más, la atención promedio, buen surtid...",NEU,0.473271,0



Ejemplos aleatorios de NEG:


,review_text_clean,sentiment_label,sentiment_score,sentiment_numeric
40706,es económico pero sinceramente nos llevamos un...,NEG,0.980127,-1
32449,llevé mi auto a este taller por una emergencia...,NEG,0.969271,-1
25473,pésimo porque el cajero no sirve,NEG,0.969185,-1
22713,"voy llegando a mi casa, estamos a punto de com...",NEG,0.759271,-1
5131,las encargadas son super groseras y despotas,NEG,0.983676,-1
15060,pongan mucho cuidado cuando pagan ya que me en...,NEG,0.837812,-1
42138,mucha fila siempre no son movidas las cajeras ...,NEG,0.961729,-1
40654,a partir de las 8 esta cerrado y no te dan tic...,NEG,0.957599,-1
14801,a los padres no se nos permite entrar a la esc...,NEG,0.975713,-1
49435,"las probé hace años, siguen siendo un chiste, ...",NEG,0.830936,-1


,review_text_clean,sentiment_label,sentiment_score,sentiment_numeric
30351,"excelente lugar. buen, bonito y barato. lo úni...",POS,0.340827,1
20207,"la verdad, no es la mejor sucursal de cinemex ...",NEG,0.345965,-1
14022,"muy buen lugar para aprender algún deporte, vo...",NEU,0.346546,0
12736,"el pollo rostizado muy rico, la atención es rá...",NEU,0.358256,0
27752,hay lo que uno necesita aunque el señor consta...,NEG,0.361169,-1
47115,hablare del pastel de elote y pan de naranja. ...,POS,0.364291,1
41815,en cuanto al servicio de los meseros muy bueno...,POS,0.364977,1
21319,es tan ricas,NEU,0.365121,0
27058,muy bonitos zapatos y variados. precios muy al...,NEU,0.365229,0
43408,todo excelente lastima q no alcanzamos mas tacos,POS,0.365762,1


In [10]:
#MÁS REVISIÓN

#Casos negativos con score alto

df_neg_high = df[
    (df["sentiment_label"] == "NEG") &
    (df["sentiment_score"] >= 0.90)
].head(20)

display(
    df_neg_high[
        ["review_text_clean", "sentiment_label", "sentiment_score"]
    ]
)


#Casos ´positivos con score alto

df_pos_high = df[
    (df["sentiment_label"] == "POS") &
    (df["sentiment_score"] >= 0.90)
].head(20)

display(
    df_pos_high[
        ["review_text_clean", "sentiment_label", "sentiment_score"]
    ]
)

#Sentimiento coincide con review rating
pd.crosstab(
    df["review_rating"],
    df["sentiment_label"],
    normalize="index"
) * 100

#Guardar

output_file = "reviews_with_transformer_sentiment.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Archivo actualizado y guardado:", output_file)

,review_text_clean,sentiment_label,sentiment_score
5,la comida ya no es buena.pedi flautas de papa ...,NEG,0.929104
8,lamentable el acoso del cajero en la hora de l...,NEG,0.959621
10,"la misoshiro sabe horrible, como si le hubiera...",NEG,0.982645
12,"pésimo servicio. pedí para llevar y, además de...",NEG,0.976635
13,consumí sopa miso con limón esa sopa no lleva ...,NEG,0.978766
22,por primera vez me invitaron a este restaurant...,NEG,0.909886
45,"siempre que vengo aquí, hay una chica que supo...",NEG,0.979145
48,el trato de casi la mayoría es bueno pero el d...,NEG,0.968534
49,"pésimo, se les junta la gente y se olvidan de ...",NEG,0.968269
54,"ya no hay gente como antes de la pandemia, aho...",NEG,0.915030


,review_text_clean,sentiment_label,sentiment_score
1,esta sucursal en especial me parece fenomenal....,POS,0.981720
2,"riquísimos y deliciosos tacos, desde pastor tr...",POS,0.943110
3,la comida es buena (a secas) tiene buen sabor ...,POS,0.949377
4,la casa del pastor se encuentra ubicado en pla...,POS,0.972359
11,"la comida exquisita a mi parecer, pedí ramen y...",POS,0.977105
15,muy rico siempre buena opción,POS,0.930683
16,"el sabor de la comida es muy rico, la comida l...",POS,0.977664
17,"la atención del personal es muy buena, todo lo...",POS,0.916681
18,"me atendieron muy rápido, son muy amables y la...",POS,0.978844
20,hace mucho no iba pero ahora que lo visité te ...,POS,0.907045


Archivo actualizado y guardado: reviews_with_transformer_sentiment.csv
